# 1.0 — Exploração e validação interativa

Notebook de apoio (o relatório oficial é `reports/relatorio.qmd`). Pressupõe `make data` (ou `make data_sim`) já executado.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd
from ib_enem_bayes.config import FEATURES, RESPONSE, SAMPLE_N, SEED, YEAR_MAIN
from ib_enem_bayes.features import processed_path, subsample
from ib_enem_bayes.modeling.frequentist import fit_mle
from ib_enem_bayes.modeling.conjugate import jeffreys_prior
from ib_enem_bayes.modeling.samplers import gibbs

In [ ]:
df = subsample(pd.read_parquet(processed_path(YEAR_MAIN)), SAMPLE_N, SEED)
y, X = df[RESPONSE].to_numpy(), df[FEATURES].to_numpy()
df.describe()

In [ ]:
mle = fit_mle(y, X)
mle.table()

In [ ]:
prior = jeffreys_prior(X.shape[1])
exact = prior.update(y, X)
draws = gibbs(y, X, prior, n_draws=2000, n_burnin=500, n_chains=4, seed=1)
pd.DataFrame({'exata': np.append(exact.m, exact.sigma2_marginal().mean()),
              'gibbs': draws.reshape(-1, 5).mean(0)}, index=FEATURES + ['sigma2'])